# Neteja i preparació dels datasets — Refugis climàtics BCN

Notebook dedicat exclusivament a documentar el procés de neteja i enriquiment de cada dataset original, un per un. Diferent del notebook d'anàlisi (`analisi_refugis_climatics_bcn.ipynb`), que parteix de les dades ja netes.

**Datasets a documentar:** 1) Refugis climàtics, 2) Població per barri, 3) Renda per barri, 4) Límits geogràfics de barri.

## 1. Refugis climàtics

**Context:** es van descarregar dues fonts independents (Generalitat de Catalunya i Ajuntament de Barcelona). Es va decidir treballar primer sobre el dataset de la Generalitat, ja que tenia els refugis categoritzats.

**Aquest script fa:**
1. Corregeix `Latitud`/`Longitud` de text a número.
2. Converteix la data d'actualització a `datetime`.
3. Tracta valors nuls a `Climatització` i `Aigua potable`.
4. Reassigna una `Categoria` neta i coherent per evitar tenir masses subgrups.
5. Afegeix `Gratuït` i `Accés Lliure`, deduïts a partir de la categoria.
6. Desa el dataset net.

**Decisions:**
- Les categories `Comerç`, `Farmàcia` i `Altres` del dataset original es van recategoritzar com a `Microrefugi` (hereten els seus valors de Gratuït/Accés Lliure: Parcialment/Parcialment).
- `Piscina` és l'única categoria amb `Gratuït = No` (les piscines municipals no són gratuïtes) — la resta de categories només tenen Sí o Parcialment.

In [ ]:
import pandas as pd
import os

BASE_DIR = os.path.join(os.path.dirname(__file__) if "__file__" in dir() else ".", "..")
INPUT_PATH = os.path.join(BASE_DIR, "1. Data", "raw", "01_refugis_climatics_raw.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "1. Data", "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
print(f"Dataset carregat: {df.shape[0]} files x {df.shape[1]} columnes")

### 1.1 Coordenades i data

In [ ]:
for col in ["Latitud", "Longitud"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(r'\.(?=\d{3})', '', regex=True)
        .astype(float)
    )

print("Coordenades convertides:")
print(f"  Latitud  -> min: {df['Latitud'].min():.5f}  max: {df['Latitud'].max():.5f}")
print(f"  Longitud -> min: {df['Longitud'].min():.5f}  max: {df['Longitud'].max():.5f}")

df["Última Actualització"] = pd.to_datetime(df["Última Actualització"], dayfirst=True)

df["Climatització"] = df["Climatització"].fillna("No informat")
df["Aigua potable"] = df["Aigua potable"].fillna("No informat")

### 1.2 Assignació de categoria

In [ ]:
def assigna_categoria(row):
    tipologia = row["Tipologia Refugi"]
    nom = str(row["Nom"])
    nom_lower = nom.lower()

    mapa_directe = {
        "_microRefugi": "Microrefugi",
        "Biblioteques": "Biblioteca",
        "Piscines": "Piscina",
        "Parcs i Jardins": "Parc i Jardí",
        "Complexos esportius": "Complex Esportiu",
        "Mercats": "Mercat",
        "Jocs d'aigua": "Joc d'Aigua",
        "Museus": "Museu",
        "Interiors d'Illa": "Interior d'Illa",
        "Patis d'Escola": "Pati d'Escola",
        "Patis d'Escola Bressol": "Pati d'Escola Bressol",
        "Centres de culte": "Centre de Culte",
        "Entitats culturals": "Entitat Cultural",
        "Comerços": "Microrefugi",
        "Centres comercials": "Centre Comercial",
        "Equipaments Ambientals": "Equipament Ambiental",
        "Equipaments de ciutat": "Equipament de Ciutat",
    }

    if tipologia in mapa_directe:
        return mapa_directe[tipologia]

    if tipologia == "Altres":
        if "col·legi de metges" in nom_lower or "col·legi oficial d'infermeres" in nom_lower:
            return "Equipament Sanitari"
        elif "foodcoop" in nom_lower:
            return "Microrefugi"  
        else:
            return "Entitat Social"

    if tipologia == "Equipaments de proximitat":

        if (nom.startswith("CC ")
                or "centre cívic" in nom_lower
                or "centre civic" in nom_lower
                or "casal cívic i comunitari" in nom_lower
                or nom.startswith("Sala ")):
            return "Centre Cívic"

        elif (nom.startswith("CGG ")
                or "casal de gent gran" in nom_lower
                or "casal gent gran" in nom_lower
                or "casal de gg" in nom_lower
                or "espai de gent gran" in nom_lower
                or "espai gent gran" in nom_lower
                or "casal de persones grans" in nom_lower):
            return "Casal de Gent Gran"

        elif ("casal de barri" in nom_lower
                or "casal barri" in nom_lower
                or "casal font" in nom_lower
                or "casal mas" in nom_lower
                or "centre de vida comunitària" in nom_lower
                or "espai veïnal" in nom_lower
                or "espai comunitari" in nom_lower
                or "centre d'atenció integral" in nom_lower
                or "la lleialtat" in nom_lower):
            return "Casal de Barri"

        elif "espai jove" in nom_lower:
            return "Espai Jove"

        elif ("hospital" in nom_lower
                or "clínica" in nom_lower
                or "clinica" in nom_lower
                or nom_lower.startswith("cap ")
                or " cap " in nom_lower
                or nom_lower.startswith("eap ")
                or "institut català de la salut" in nom_lower):
            return "Equipament Sanitari"

        elif "farmàcia" in nom_lower or "farmacia" in nom_lower:
            return "Microrefugi" 

        elif nom.startswith("CPG "):
            return "Complex Esportiu"

        elif ("seu districte" in nom_lower
                or "seu del districte" in nom_lower
                or "equipament municipal" in nom_lower
                or "torre jussana" in nom_lower
                or "castell de torre baró" in nom_lower
                or "centre d'informació del parc" in nom_lower
                or "la masia" in nom_lower):
            return "Equipament de Ciutat"

        elif ("centre cultural" in nom_lower
                or "fabra i coats" in nom_lower
                or "centre ton i guida" in nom_lower
                or "espai montserrat" in nom_lower):
            return "Entitat Cultural"

        elif ("fundació" in nom_lower
                or "creu roja" in nom_lower
                or "punt d'activitat" in nom_lower
                or "casa asil" in nom_lower):
            return "Entitat Social"

        elif "casa de l'aigua" in nom_lower:
            return "Equipament Ambiental"

        else:
            return "Equipament de Ciutat"

    return "Microrefugi" 


df["Categoria"] = df.apply(assigna_categoria, axis=1)

print("Categories assignades:")
print(df["Categoria"].value_counts().to_string())

### 1.3 Gratuït i Accés Lliure

Per a poder fer un bon filtrat dels refugis al anàlisi posterior, necessitem determinar quines categories de refugi compleixen ser gratuïtes i d'accés lliure.


Deduïts a partir de la `Categoria`. Els valors "Parcialment" no signifiquen el mateix a totes dues columnes — cadascuna captura un matís diferent:

**Gratuït:**
- **Sí** — accés sense cap cost.
- **Parcialment** — establiments amb un incentiu comercial (mercats, museus que cobren entrada, complexos esportius amb qüota, centres comercials, microrefugis en comerços privats...) que no exigeixen pagament per accedir-hi o fer-hi ús com a refugi. No vol dir "meitat de preu", sinó "hi ha un context comercial, però no cal comprar per refugiar-s'hi".
- **No** — exclusiu de `Piscina`: cal pagar entrada per accedir-hi, sense excepció.

**Accés Lliure:**
- **Sí** — es pot romandre l'espai el temps que faci falta i s'ofereixen activitats.
- **Parcialment** — limitació de temps d'ús (refrescar-se una estona puntual), no una restricció d'entrada. El lloc és un refugi climàtic vàlid, però no pensat per passar-hi hores.
- **Restringit** — accés condicionat (p. ex. cal ser usuari habitual o soci de l'equipament si es vol fer ús continuat del espai).

Aquesta distinció és deliberada, no un valor ambigu: reflecteix diferències reals en el tipus de refugi (un mercat i una biblioteca són tots dos "gratuïts", però amb lògiques d'ús molt diferents).

In [1]:
mapa_gratuit = {
    "Microrefugi": "Parcialment",
    "Biblioteca": "Sí",
    "Centre Cívic": "Sí",
    "Casal de Barri": "Sí",
    "Casal de Gent Gran": "Sí",
    "Espai Jove": "Sí",
    "Mercat": "Parcialment",
    "Museu": "Parcialment",
    "Entitat Cultural": "Sí",
    "Entitat Social": "Sí",
    "Centre de Culte": "Sí",
    "Parc i Jardí": "Sí",
    "Interior d'Illa": "Sí",
    "Joc d'Aigua": "Sí",
    "Pati d'Escola": "Sí",
    "Pati d'Escola Bressol": "Sí",
    "Piscina": "No",
    "Complex Esportiu": "Parcialment",
    "Centre Comercial": "Parcialment",
    "Equipament Sanitari": "Sí",
    "Equipament Ambiental": "Sí",
    "Equipament de Ciutat": "Sí",
}

mapa_acces_lliure = {
    "Microrefugi": "Parcialment",
    "Biblioteca": "Sí",
    "Centre Cívic": "Sí",
    "Casal de Barri": "Sí",
    "Casal de Gent Gran": "Restringit",
    "Espai Jove": "Restringit",
    "Mercat": "Sí",
    "Museu": "Sí",
    "Entitat Cultural": "Sí",
    "Entitat Social": "Parcialment",
    "Centre de Culte": "Parcialment",
    "Parc i Jardí": "Sí",
    "Interior d'Illa": "Sí",
    "Joc d'Aigua": "Sí",
    "Pati d'Escola": "Restringit",
    "Pati d'Escola Bressol": "Restringit",
    "Piscina": "Sí",
    "Complex Esportiu": "Sí",
    "Centre Comercial": "Sí",
    "Equipament Sanitari": "Parcialment",
    "Equipament Ambiental": "Sí",
    "Equipament de Ciutat": "Parcialment",
}

df["Gratuït"] = df["Categoria"].map(mapa_gratuit)
df["Accés Lliure"] = df["Categoria"].map(mapa_acces_lliure)

assert df["Gratuït"].isnull().sum() == 0, "Hi ha categories sense valor de Gratuït assignat"
assert df["Accés Lliure"].isnull().sum() == 0, "Hi ha categories sense valor d'Accés Lliure assignat"

print("Gratuït:")
print(df["Gratuït"].value_counts().to_string())
print("\nAccés Lliure:")
print(df["Accés Lliure"].value_counts().to_string())

NameError: name 'df' is not defined

### 1.4 Comprovacions finals i desar

In [ ]:
nuls_restants = df.isnull().sum()
nuls_restants = nuls_restants[nuls_restants > 0]
if len(nuls_restants) > 0:
    print("Nuls restants:")
    print(nuls_restants.to_string())
else:
    print("Cap valor nul a les columnes principals")

output_path = os.path.join(OUTPUT_DIR, "01_refugis_climatics_clean.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\nDataset net desat: {df.shape[0]} files x {df.shape[1]} columnes")

### 1.5 Fusió amb dataset actualitzat d'Open Data BCN (detecció de registres nous)

Pas posterior a 1.1-1.4: l'Ajuntament va publicar una versió actualitzada del llistat de refugis (`opendatabcn_NP-NASIA_xarxa-refugis-climatics-csv.csv`). Cal detectar quins registres ja existien al dataset net i quins són genuïnament nous, per no duplicar ni perdre informació.

**Mètode: matching per distància haversina**, no per nom ni ID (els identificadors no són consistents entre fonts). Es calcula la distància en metres entre cada refugi del dataset actual i el més proper del dataset nou; per sota d'un llindar de **30 metres** es considera el mateix lloc físic.

**Detall tècnic important:** el dataset actual tenia un bug d'emmagatzematge a `Latitud`/`Longitud` (es va perdre el punt decimal, p.ex. `4137923.0` en lloc de `41.37923`) — `fix_lat`/`fix_lon` el corregeixen reconstruint el punt a la posició correcta abans de calcular cap distància.

**Resultat del matching automàtic:** 123 registres del dataset nou no van trobar cap coincidència a &lt;= 30 m (`candidats_nous.csv`).

**Revisió manual (no reproduïble en codi):** dels 123 candidats, la majoria ja existien al dataset (coincidències que el llindar de 30 m no va detectar automàticament — probablement per petites discrepàncies de coordenades entre fonts). Després de la revisió manual, **només 27 registres eren genuïnament nous**, principalment **comerços privats**.


In [ ]:
import numpy as np

RUTA_ACTUAL = os.path.join(OUTPUT_DIR, "01_refugis_climatics_clean.csv")
RUTA_NOU = os.path.join(BASE_DIR, "1. Data", "raw", "opendatabcn_NP-NASIA_xarxa-refugis-climatics-csv.csv")
LLINDAR_M = 30

old = pd.read_csv(RUTA_ACTUAL, dtype=str)
old.columns = [c.strip().lstrip("\ufeff") for c in old.columns]


def fix_lat(s):
    s2 = s[:-2] if s.endswith(".0") else s.replace(".", "")
    return round(float(s2[:2] + "." + s2[2:]), 6)


def fix_lon(s):
    s2 = s[:-2] if s.endswith(".0") else s.replace(".", "")
    return round(float(s2[:1] + "." + s2[1:]), 6)


old["Latitud"] = old["Latitud"].apply(fix_lat)
old["Longitud"] = old["Longitud"].apply(fix_lon)

new = pd.read_csv(RUTA_NOU, dtype=str, encoding="utf-16")
new.columns = [c.strip().lstrip("\ufeff") for c in new.columns]
new["register_id"] = new["register_id"].str.lstrip("\ufeff")

new_clean = pd.DataFrame(
    {
        "id_nou": range(1, len(new) + 1),
        "register_id": new["register_id"],
        "Nom": new["name"],
        "Adreça": new["addresses_road_name"].fillna("") + ", " + new["addresses_start_street_number"].fillna(""),
        "Barri": new["addresses_neighborhood_name"],
        "Districte": new["addresses_district_name"],
        "Latitud": new["geo_epgs_4326_lat"].astype(float).round(6),
        "Longitud": new["geo_epgs_4326_lon"].astype(float).round(6),
    }
)

R = 6371000

lat1 = np.radians(old["Latitud"].values)[:, None]
lon1 = np.radians(old["Longitud"].values)[:, None]
lat2 = np.radians(new_clean["Latitud"].values)[None, :]
lon2 = np.radians(new_clean["Longitud"].values)[None, :]

dlat = lat2 - lat1
dlon = lon2 - lon1
a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
dist = 2 * R * np.arcsin(np.sqrt(a))

nn_new_idx = dist.argmin(axis=1)
nn_new_dist = dist.min(axis=1)
nn_old_dist = dist.min(axis=0)

matching = old[["Nom", "Categoria", "Adreça", "Latitud", "Longitud"]].copy()
matching["Coincidencia"] = np.where(nn_new_dist <= LLINDAR_M, "Coincident", "Només al dataset actual")
matching["Nom_nou_relacionat"] = np.where(nn_new_dist <= LLINDAR_M, new_clean["Nom"].values[nn_new_idx], "")
matching["Distancia_m"] = nn_new_dist.round(1)

candidats_nous = new_clean.loc[
    nn_old_dist > LLINDAR_M, ["id_nou", "Nom", "Adreça", "Barri", "Districte", "Latitud", "Longitud"]
].copy()
candidats_nous["Distancia_al_mes_proper_actual_m"] = nn_old_dist[nn_old_dist > LLINDAR_M].round(1)


n_coincident = int((matching["Coincidencia"] == "Coincident").sum())
n_only_old = len(old) - n_coincident
n_only_new = len(candidats_nous)

print(f"Registres dataset actual: {len(old)}")
print(f"Registres dataset nou:    {len(new_clean)}")
print(f"Coincidents (<= {LLINDAR_M} m): {n_coincident}")
print(f"Només al dataset actual:  {n_only_old}")
print(f"Candidats nous (abans de revisio manual): {n_only_new}")
print("-> Despres de revisio manual: nomes 27 eren genuinament nous (principalment comercos privats).")

matching.to_csv(os.path.join(OUTPUT_DIR, "matching_complet.csv"), index=False, encoding="utf-8-sig")
candidats_nous.to_csv(os.path.join(OUTPUT_DIR, "candidats_nous.csv"), index=False, encoding="utf-8-sig")

## 2. Població per barri

**Font:** padró municipal per barri, edat individual i sexe (2026) - Open Data Barcelona

**Aquest script fa:**
1. Converteix els valors confidencials (`..`, dades de baixa fiabilitat per volum petit) a 0.
2. Agrega de 46.238 files (barri x edat x sexe) a 73 files (una per barri): població total, població 65+ i població <5, en valor absolut i percentatge.
3. Desa el dataset net.

In [ ]:
INPUT_PATH_POB = os.path.join(BASE_DIR, "1. Data", "raw", "02_poblacio_barri_raw.csv")

df_pob = pd.read_csv(INPUT_PATH_POB)
print(f"Dataset carregat: {df_pob.shape[0]} files x {df_pob.shape[1]} columnes")

df_pob["Valor"] = pd.to_numeric(df_pob["Valor"], errors="coerce").fillna(0).astype(int)
print("Valors confidencials ('..') convertits a 0")

resum = df_pob.groupby(["Codi_Barri", "Nom_Barri", "Nom_Districte"]).apply(
    lambda x: pd.Series({
        "poblacio_total": x["Valor"].sum(),
        "poblacio_65_mes": x[x["EDAT_1"] >= 65]["Valor"].sum(),
        "poblacio_menys_5": x[x["EDAT_1"] < 5]["Valor"].sum(),
    })
).reset_index()

resum["pct_65_mes"] = (resum["poblacio_65_mes"] / resum["poblacio_total"] * 100).round(2)
resum["pct_menys_5"] = (resum["poblacio_menys_5"] / resum["poblacio_total"] * 100).round(2)

print(f"\nBarris: {len(resum)}")
print(f"Poblacio total BCN: {resum['poblacio_total'].sum():,}")

print(f"\nMajors de 65 anys -- min: {resum['pct_65_mes'].min():.1f}%  max: {resum['pct_65_mes'].max():.1f}%  mitjana: {resum['pct_65_mes'].mean():.1f}%")
print(f"Menors de 5 anys  -- min: {resum['pct_menys_5'].min():.1f}%  max: {resum['pct_menys_5'].max():.1f}%  mitjana: {resum['pct_menys_5'].mean():.1f}%")

print("\nTop 5 barris amb mes gent gran:")
print(resum.nlargest(5, "pct_65_mes")[["Nom_Barri", "pct_65_mes"]].to_string(index=False))

print("\nTop 5 barris amb mes menors de 5 anys:")
print(resum.nlargest(5, "pct_menys_5")[["Nom_Barri", "pct_menys_5"]].to_string(index=False))

# --- Desar ---
output_path_pob = os.path.join(OUTPUT_DIR, "02_poblacio_barri_clean.csv")
resum.to_csv(output_path_pob, index=False, encoding="utf-8-sig")
print(f"\nDataset net desat: {resum.shape[0]} files x {resum.shape[1]} columnes")

## 3. Renda per barri

**Font:** Atles de Distribució de Renda de les Llars (INE), agregat per l'Ajuntament de Barcelona. Nivell original: secció censal (1.068 files, 73 barris, 10 districtes).

**Aquest script fa:** agrega la renda bruta per llar de secció censal a barri.

**Decisions clau:**
- Clau d'agregació: `Codi_Barri` (codi oficial 1-73), no `Nom_Barri` — evita discrepàncies d'estil de text entre fonts.
- Mètode: **mediana**, no mitjana, per minimitzar l'efecte d'outliers dins un mateix barri.
- **Limitació explícita:** no es disposa de població per secció censal, per tant no es pot fer una mitjana ponderada real. 3 barris (la Marina del Prat Vermell, la Clota, Vallbona) tenen només 1 secció censal — la seva "mediana" és, de fet, el valor d'una única observació.

In [ ]:
INPUT_PATH_RENDA = os.path.join(BASE_DIR, "1. Data", "raw", "03_renda_bruta_llar_2023_raw.csv")

renda_raw = pd.read_csv(INPUT_PATH_RENDA)

renda_barri = (
    renda_raw.groupby("Codi_Barri", as_index=False)
    .agg(
        Nom_Barri=("Nom_Barri", "first"),
        Nom_Districte=("Nom_Districte", "first"),
        n_seccions=("Seccio_Censal", "count"),
        Renda_Bruta_Mediana=("Import_Renda_Bruta_€", "median"),
    )
    .sort_values("Codi_Barri")
    .reset_index(drop=True)
)

assert renda_barri.shape[0] == 73, f"S'esperaven 73 barris, se n'han obtingut {renda_barri.shape[0]}"
assert renda_barri["Codi_Barri"].is_unique, "Codi_Barri te valors duplicats"
assert renda_barri.isnull().sum().sum() == 0, "Hi ha valors nuls al resultat agregat"

output_path_renda = os.path.join(OUTPUT_DIR, "03_renda_bruta_llar_2023_clean.csv")
renda_barri.to_csv(output_path_renda, index=False, encoding="utf-8")

print(f"Agregacio completada: {renda_barri.shape[0]} barris.")
print("Barris amb nomes 1 seccio censal (mediana d'una sola observacio):")
print(renda_barri.loc[renda_barri["n_seccions"] == 1, ["Codi_Barri", "Nom_Barri"]].to_string(index=False))

## 4. Límits geogràfics de barri (GeoJSON) i join espacial amb els refugis

**Font:** GeoJSON oficial de barris de Barcelona (Open Data BCN / mirall martgnz/bcn-geodata), amb camps `BARRI` (= `Codi_Barri`), `NOM` i `DISTRICTE`.

**Aquest script fa:**
1. Converteix els refugis (Latitud, Longitud) en punts geogràfics.
2. Point-in-polygon: per a cada refugi, determina dins de quin polígon de barri cau i li assigna `Codi_Barri`.
3. **Casos límit:** refugis que no encaixen a cap polígon (frontera/error GPS) es reassignen al barri més proper amb `sjoin_nearest`; es comprova que no hi hagi refugis duplicats (p. ex. per caure just a sobre una frontera compartida).
4. Desa el resultat enriquit.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

REFUGIS_PATH_JOIN = os.path.join(OUTPUT_DIR, "01_refugis_climatics_clean.csv")
BARRIS_PATH = os.path.join(OUTPUT_DIR, "04_barris_geo_espacial.geojson")
OUTPUT_PATH_JOIN = os.path.join(OUTPUT_DIR, "01_refugis_climatics_amb_barri.csv")

refugis_join = pd.read_csv(REFUGIS_PATH_JOIN)
geometria_punts = [Point(lon, lat) for lon, lat in zip(refugis_join["Longitud"], refugis_join["Latitud"])]
refugis_gdf = gpd.GeoDataFrame(refugis_join, geometry=geometria_punts, crs="EPSG:4326")

barris_gdf = gpd.read_file(BARRIS_PATH)
barris_gdf = barris_gdf[["BARRI", "NOM", "DISTRICTE", "geometry"]].rename(
    columns={"BARRI": "Codi_Barri", "NOM": "Nom_Barri", "DISTRICTE": "Codi_Districte"}
)
barris_gdf["Codi_Barri"] = barris_gdf["Codi_Barri"].astype(int)

resultat = gpd.sjoin(refugis_gdf, barris_gdf, how="left", predicate="within")

sense_barri = resultat[resultat["Codi_Barri"].isna()]
print(f"Refugis sense barri assignat (within): {len(sense_barri)}")
if len(sense_barri) > 0:
    refugis_sense_barri = refugis_gdf.loc[sense_barri.index]
    reassignats = gpd.sjoin_nearest(refugis_sense_barri, barris_gdf, how="left")
    resultat.loc[sense_barri.index, ["Codi_Barri", "Nom_Barri", "Codi_Districte"]] = (
        reassignats[["Codi_Barri", "Nom_Barri", "Codi_Districte"]].values
    )
    print("  -> reassignats al barri mes proper amb sjoin_nearest")

n_original = len(refugis_join)
n_resultat = len(resultat)
print(f"Files originals: {n_original} | Files despres del join: {n_resultat}")
if n_resultat > n_original:
    print("ATENCIO: hi ha refugis duplicats (probablement per caure sobre una frontera).")
    duplicats = resultat[resultat.duplicated(subset=["Nom", "Latitud", "Longitud"], keep=False)]
    print(duplicats[["Nom", "Latitud", "Longitud", "Codi_Barri", "Nom_Barri"]])

assert resultat["Codi_Barri"].isna().sum() == 0, "Encara queden refugis sense Codi_Barri"

columnes_finals = list(refugis_join.columns) + ["Codi_Barri", "Nom_Barri", "Codi_Districte"]
resultat[columnes_finals].to_csv(OUTPUT_PATH_JOIN, index=False, encoding="utf-8")

print(f"\nFet. Guardat a {OUTPUT_PATH_JOIN}")
print(resultat["Nom_Barri"].value_counts().head(10))